# Tech Challenge Fase 1 — Diagnóstico de Diabetes
## 1. Exploração de Dados (EDA)
Este notebook realiza a análise exploratória do dataset Pima Indians Diabetes,
identificando características, distribuições e problemas de qualidade nos dados.

## 1.1 Importação de bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações visuais
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', None)

print('Bibliotecas importadas com sucesso!')

## 1.2 Carregamento do dataset

In [ ]:
df = pd.read_csv('../data/diabetes.csv')
print(f'Dataset carregado: {df.shape[0]} linhas e {df.shape[1]} colunas')
df.head(10)

## 1.3 Informações gerais do dataset

In [ ]:
print('=== INFORMAÇÕES GERAIS ===')
df.info()

In [ ]:
print('=== ESTATÍSTICAS DESCRITIVAS ===')
df.describe().round(2)

## 1.4 Verificação de valores ausentes

In [ ]:
print('=== VALORES NULOS ===')
print(df.isnull().sum())
print(f'\nTotal de nulos: {df.isnull().sum().sum()}')

## 1.5 Zeros suspeitos (dados ausentes disfarçados)

Algumas colunas não podem ter valor zero fisiologicamente:
- **Glucose**: glicose zero é incompatível com vida
- **BloodPressure**: pressão zero é incompatível com vida  
- **SkinThickness**: espessura de pele zero não faz sentido clínico
- **Insulin**: pode ser zero mas é suspeito
- **BMI**: IMC zero é impossível

Esses zeros provavelmente representam dados não coletados (ausentes disfarçados).

In [ ]:
colunas_suspeitas = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

print('=== ZEROS SUSPEITOS POR COLUNA ===')
for col in colunas_suspeitas:
    zeros = (df[col] == 0).sum()
    pct = (zeros / len(df)) * 100
    print(f'{col}: {zeros} zeros ({pct:.1f}%)')

## 1.6 Balanceamento da variável alvo (Outcome)

In [ ]:
print('=== DISTRIBUIÇÃO DA CLASSE OUTCOME ===')
contagem = df['Outcome'].value_counts()
pct = df['Outcome'].value_counts(normalize=True) * 100

print(f'Sem diabetes (0): {contagem[0]} pacientes ({pct[0]:.1f}%)')
print(f'Com diabetes (1): {contagem[1]} pacientes ({pct[1]:.1f}%)')

# Gráfico
fig, ax = plt.subplots()
bars = ax.bar(['Sem Diabetes (0)', 'Com Diabetes (1)'], contagem.values, color=['steelblue', 'tomato'])
ax.set_title('Distribuição da Variável Alvo (Outcome)', fontsize=14)
ax.set_ylabel('Quantidade de Pacientes')
for bar, val in zip(bars, contagem.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(val), ha='center', fontsize=12)
plt.tight_layout()
plt.savefig('../notebooks/outcome_distribuicao.png', dpi=150)
plt.show()
print('\nObservação: dataset desbalanceado — isso será considerado na modelagem.')

## 1.7 Distribuição das features

In [ ]:
features = [col for col in df.columns if col != 'Outcome']

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(features):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=12)
    axes[i].set_ylabel('Frequência')

plt.suptitle('Distribuição das Features', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('../notebooks/features_distribuicao.png', dpi=150)
plt.show()

## 1.8 Distribuição das features por classe (Outcome)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(features):
    for outcome, color, label in [(0, 'steelblue', 'Sem Diabetes'), (1, 'tomato', 'Com Diabetes')]:
        axes[i].hist(df[df['Outcome'] == outcome][col], bins=25, alpha=0.6, color=color, label=label, edgecolor='white')
    axes[i].set_title(col, fontsize=12)
    axes[i].legend(fontsize=8)

plt.suptitle('Distribuição das Features por Classe', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('../notebooks/features_por_classe.png', dpi=150)
plt.show()

## 1.9 Análise de correlação

In [ ]:
correlacao = df.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(
    correlacao,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5
)
plt.title('Mapa de Correlação entre Features', fontsize=14)
plt.tight_layout()
plt.savefig('../notebooks/correlacao_heatmap.png', dpi=150)
plt.show()

print('\n=== CORRELAÇÃO COM OUTCOME (ordenada) ===')
print(correlacao['Outcome'].drop('Outcome').sort_values(ascending=False).round(3))

## 1.10 Conclusões da EDA

**Resumo dos achados:**

1. **Dataset:** 768 pacientes, 8 features numéricas + 1 variável alvo (Outcome)
2. **Valores ausentes:** nenhum valor nulo explícito, porém há zeros suspeitos em colunas como Glucose, BloodPressure, SkinThickness, Insulin e BMI
3. **Desbalanceamento:** ~65% sem diabetes (0) e ~35% com diabetes (1) — dataset moderadamente desbalanceado
4. **Features mais correlacionadas com Outcome:** Glucose apresenta a maior correlação positiva, seguida de BMI e Age
5. **Próximo passo:** pré-processamento (tratar zeros suspeitos, escalar features, separar treino/teste)
